<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 컴퓨터 비전 · 08. Vision Transformer (ViT)

## An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale

- **원 논문:** [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)
- **저자 / 발표:** Alexey Dosovitskiy et al. · ICLR 2021 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** patch embedding, class token, position embedding, pre-LN encoder를 재현한다.

**축소하거나 생략한 부분:** 224×224/patch16, 대규모 사전학습 대신 16×16/patch4와 1 encoder layer를 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Fig. 1: patches as tokens | Task 1 patchify | 16 patches |
| §3.1 Eq. (1–4): class/position embedding and LN/MSA/MLP residual | Task 2 explicit Q/K/V pre-LN encoder | 17 tokens and [B,H,17,17] attention weights |
| §3.2: fine-tuning | Task 3 | loss/accuracy |

## 핵심 아이디어와 수식

$$\mathbf{z}_0=[\mathbf{x}_{class};\mathbf{x}_p^1E;\cdots;\mathbf{x}_p^NE]+E_{pos}$$

**기호 해설:** x_p는 펼친 patch, E는 projection, x_class는 학습 token, E_pos는 위치 embedding이다.

**코드 대응:** Task 1이 patchify/config, Task 2가 class token과 Q/K/V projection, head 분할, pre-LN residual encoder를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → patches [B,16,16] → tokens [B,17,24] → logits [B,4].

**포트폴리오 구현 범위:** 대규모 사전학습과 hybrid CNN은 제외하지만 tokenization과 encoder 식은 유지한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Fig. 1: patches as tokens
- **노트북에서 구현할 부분:** Task 1 patchify
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 16 patches

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 1: ViT 설정과 unfold를 이용한 patchify를 구현하세요.
@dataclass
class ViTConfig:
    pass

    @property
    def patch_count(self):
        """이미지를 분할할 겹치지 않는 patch 개수를 반환한다."""
        raise NotImplementedError


def patchify(images, patch_size):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (1–4): class/position embedding and LN/MSA/MLP residual
- **노트북에서 구현할 부분:** Task 2 explicit Q/K/V pre-LN encoder
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 17 tokens and [B,H,17,17] attention weights

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 2: class token, position embedding, pre-LN residual encoder를 구현하세요.
class ExplicitMultiHeadAttention(nn.Module):
    """ViT Eq. (2)에 사용하는 multi-head self-attention이다."""

    def __init__(self, model_dim, heads):
        super().__init__()

    def split_heads(self, tensor):
        """[B, T, D]를 [B, H, T, d_h]로 분할한다."""
        raise NotImplementedError

    def merge_heads(self, tensor):
        """[B, H, T, d_h]를 [B, T, D]로 병합한다."""
        raise NotImplementedError

    def forward(self, query, key, value, attention_mask=None):
        """Attention output과 weights [B, H, T_q, T_k]를 반환한다."""
        raise NotImplementedError


class TransformerEncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

    def forward(self, tokens):
        raise NotImplementedError


class VisionTransformerMini(nn.Module):
    def __init__(self, config):
        super().__init__()

    def embed(self, inputs):
        """[B, 1, H, W]를 [B, N+1, D] token sequence로 변환한다."""
        raise NotImplementedError

    def forward(self, inputs):
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.2: fine-tuning
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** loss/accuracy

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
# TODO 3: 분류 손실, 한 학습 step, 평가 함수를 각각 구현하세요.
def classification_loss(logits, targets):
    raise NotImplementedError


def train_step(model, optimizer, inputs, targets):
    raise NotImplementedError


def evaluate_classifier(model, inputs, targets):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.